In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("m3_construction.ipynb")

# M3 — Bounds, a hybrid, and what your comparison licenses

**TC6035 Part 1 · Milestone 3 · final**

The last milestone is the only one that asks a question the previous three could
not: **what does your evidence actually support?**

> **Must cite M2**: your best PSO configuration and the evidence that selected it.

> A hybrid that loses to its own components, analysed honestly, scores above one
> that wins without explanation. A non-significant difference reported as
> "not measured" scores above one reported as a tie.

In [ ]:
import itertools
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon

from tc6035 import build_instance, SensorPlacementProblem
from tc6035.runlog import RunSet, save_runset, load_runset
from solutions import ALGORITHMS

STUDENT_ID = "A01234567"   # <-- yours
SALT = 0
BUDGET, N_SEEDS = 5_000, 30
instance = build_instance(STUDENT_ID, salt=SALT)

# --- carried forward from M2 -------------------------------------------------
PSO_TOPOLOGY = "ring"                 # M2 Question 1, selected by the paired test
PSO_W, PSO_C1, PSO_C2, PSO_SWARM = 0.7298, 1.4962, 1.4962, 20

# Teams: your partner's ID. Leave None if you are working alone.
PARTNER_ID = None

print(f"M2 carried forward: topology {PSO_TOPOLOGY}, swarm {PSO_SWARM}")

---
## Question 1 — Are your bounds binding?

Run your ACO twice on the same seeds: once with MMAS bounds, once without.

Compare them on **selection entropy** and **distinct subsets constructed per
iteration** — not on final objective value. A collapsed colony still reports a
respectable number, which is exactly why objective values cannot show this.

Store `entropy_bounded`, `entropy_unbounded` and `distinct_unbounded`.

In [ ]:
from tc6035.reference import selection_entropy   # provided: entropy of a pheromone vector

def run_aco(seed, use_bounds):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    return ALGORITHMS["aco"](prob, seed, use_bounds=use_bounds, track=True), prob

# The reference implementation returns per-iteration diagnostics when track=True.
bounded, unbounded = [], []
...
    ...
        prob = ...
        ...
        r = ...
        ...

entropy_bounded   = ...
entropy_unbounded = ...
distinct_unbounded = ...
distinct_bounded   = ...
ratio_unbounded    = ...

print(f"  entropy   bounded {entropy_bounded:.3f}   unbounded {entropy_unbounded:.3f}")
print(f"  distinct  bounded {distinct_bounded:.1f}     unbounded {distinct_unbounded:.1f}")
print(f"  tau_max/tau_min unbounded: {ratio_unbounded:.3e}")
print(f"  sites at tau_min (bounded): {bounded[0][-1]['at_min']} of 80")
print(f"  sites at tau_max (bounded): {bounded[0][-1]['at_max']} of 80")

In [ ]:
grader.check("q1_bounds")

---
## Question 2 — A hybrid, with the rationale stated first

Implement `hybrid(problem, seed, **params)` in `src/solutions.py`, combining at
least two approaches.

**Write your design rationale in the markdown cell below before you look at the
result.** That ordering is the point: a rationale written after the fact is
indistinguishable from a rationalization.

::: One budget. Use `problem.limited(n)` to carve out a stage:

```python
with problem.limited(int(problem.budget * split)):
    subset = ...            # stage one stops here
tune_power(problem, seed, subset)   # spends the remainder
```

Creating a second `SensorPlacementProblem` for stage one silently doubles your
budget. The autograder compares your declared evaluation count against your
recorded trajectory length, so it will not pass.

In [ ]:
HYBRID_SPLIT = ...

hyb_records, hyb_scores = [], []
for seed in range(N_SEEDS):
    prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
    ...
    hyb_records.append(prob.record)
hyb_scores = np.array(hyb_scores)

save_runset("results/m3_hybrid.npz", RunSet(algorithm="hybrid", student_id=STUDENT_ID,
            params={"split": HYBRID_SPLIT}, records=hyb_records))

print(f"hybrid median {np.median(hyb_scores):.4f}  "
      f"IQR [{np.percentile(hyb_scores,25):.4f}, {np.percentile(hyb_scores,75):.4f}]")
print(f"budget used per run: min {min(r.evaluations_used for r in hyb_records)}, "
      f"max {max(r.evaluations_used for r in hyb_records)} of {BUDGET}")

In [ ]:
grader.check("q2_hybrid")

<!-- BEGIN QUESTION -->

**Before looking at the result above**: why this combination, and what do you
expect? Name which method handles which layer and why.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — The comparative study

All your algorithms, `N_SEEDS` paired seeds. Friedman first to establish that
they differ at all, then pairwise Wilcoxon with **Holm correction**.

Store `friedman_p`, `holm_adjusted` (a dict keyed by algorithm pair) and
`n_significant`.

In [ ]:
NAMES = ["monte_carlo", "simulated_annealing", "tabu_search", "pso", "aco", "hybrid"]
KW = {"pso": {"topology": PSO_TOPOLOGY, "swarm_size": PSO_SWARM},
      "hybrid": {"split": HYBRID_SPLIT}}

scores = {}
for name in NAMES:
    vals = []
    for seed in range(N_SEEDS):
        prob = SensorPlacementProblem(instance, budget=BUDGET, seed=seed)
        vals.append(ALGORITHMS[name](prob, seed, **KW.get(name, {})))
    scores[name] = np.array(vals)

friedman_stat, friedman_p = ...

pairs = ...
raw_p = ...
                        alternative= ...
...
order = ...
m, running, holm_adjusted = ...
...
    running = ...
    holm_adjusted[pair] = ...
n_significant = ...

ranks = np.argsort(np.argsort(-np.column_stack([scores[n] for n in NAMES]), axis=1), axis=1) + 1
print(f"  {'algorithm':22} {'median':>8} {'mean rank':>10}")
for i, n in enumerate(NAMES):
    print(f"  {n:22} {np.median(scores[n]):8.4f} {ranks[:,i].mean():10.2f}")
print(f"\n  Friedman chi2={friedman_stat:.1f}  p={friedman_p:.3e}")
print(f"  significant after Holm: {n_significant} of {m}")
print("\n  NOT significant (these orderings are unmeasured, not ties):")
for pair, v in holm_adjusted.items():
    if v >= 0.05:
        print(f"    {pair[0]} vs {pair[1]}: Holm p={v:.3f}")

In [ ]:
grader.check("q3_comparative")

<!-- BEGIN QUESTION -->

---
## Question 4 — What does this license?

**Teams:** run the identical study on your partner's instance and compare the
two rankings. Does the ordering transfer?

**Individuals:** you cannot answer the transfer question with one instance — say
so explicitly, and instead state what a second instance would have to show to
change your conclusion.

Everyone: state precisely what your data licenses and what it does not. Connect
it to the no-free-lunch theorem as stated in Session 1 — including the
precondition the popular version drops.

In [ ]:
if PARTNER_ID:
    partner = build_instance(PARTNER_ID)
    partner_scores = {}
    for name in NAMES:
        vals = []
        for seed in range(N_SEEDS):
            prob = SensorPlacementProblem(partner, budget=BUDGET, seed=seed)
            vals.append(ALGORITHMS[name](prob, seed, **KW.get(name, {})))
        partner_scores[name] = np.array(vals)

    rank_a = sorted(NAMES, key=lambda n: -np.median(scores[n]))
    rank_b = sorted(NAMES, key=lambda n: -np.median(partner_scores[n]))
    print(f"  ranking, {STUDENT_ID}: {' > '.join(rank_a)}")
    print(f"  ranking, {PARTNER_ID}: {' > '.join(rank_b)}")
    print(f"  identical: {rank_a == rank_b}")
    if rank_a != rank_b:
        moved = [n for n in NAMES if rank_a.index(n) != rank_b.index(n)]
        print(f"  moved: {', '.join(moved)}")
        print("  Check whether those pairs were significant on your own instance.")
else:
    print("  Individual submission: the transfer question is not answerable here.")
    print("  Say so in your analysis rather than implying generality you cannot test.")

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Before you submit

- [ ] `ALGORITHMS` includes all six, `hybrid` among them
- [ ] `results/` holds every run log, one per algorithm
- [ ] This notebook cites **M2** by name
- [ ] `AI_LEDGER.md` section 3 documents a real error with evidence
- [ ] Every member has recorded a video and linked it in `TEAM.md`
- [ ] `python scripts/self_check.py` passes